# End-to-End RAG Flow

This notebook exercises the basic factory path for the RAG system from config loading through generation.

In [1]:
from notebook_support import setup_notebook

setup_notebook()


print("Notebook environment ready")

✓ Notebook environment setup completed successfully
✓ Project root: F:\Github\Personal\real-world-rag-system
Notebook environment ready


In [2]:
from src.config import settings
from src.config.loader import load_config
from src.utils.logger import Logger
from src.rag_components.data_loaders import DataLoaderFactory
from src.rag_components.chunkers import ChunkingFactory
from src.rag_components.embeddings.factory import EmbedderFactory
from src.rag_components.vector_stores import VectorStoreFactory
from src.rag_components.retrievers import RetrieverFactory
from src.rag_components.generators import GeneratorFactory
from src.rag_components.system_prompts import BASIC_RAG_PROMPT, ANOTHER_RAG_PROMPT


logger = Logger()
print("Imports loaded")

Project root: F:\Github\Personal\real-world-rag-system
Logger directory: F:\Github\Personal\real-world-rag-system\logs
2026-07-03 23:33:14,213 | INFO	| real_world_rag | RAG System started
Imports loaded


In [3]:
config_path = settings.pipeline_config_file
config = load_config(config_path)

print(f"Loaded config: {config_path}")
print(f"Data loader: {config.data_loader}")
print(f"Chunking: {config.chunking}")
print(f"Embedding: {config.embedding}")
print(f"Vector store: {config.vector_store}")
print(f"Retrieval: {config.retrieval}")
print(f"Generator: {config.generator}")

Loaded config: src/rag_flow_config/offline_data.yaml
Data loader: source=<DataLoaderSource.LOCAL: 'local'> path=None dataset_name='suniltvl/ragbench' subset='emanual' split='validation' cache_dir=None data_dir='./data/rag_bench' streaming=True file_extension='parquet' base_url=None
Chunking: strategy=<ChunkingStrategy.RECURSIVE: 'recursive'> chunk_size=512 chunk_overlap=100
Embedding: provider=<EmbeddingProvider.HUGGINGFACE: 'huggingface'> model='BAAI/bge-small-en-v1.5'
Vector store: provider=<VectorStoreProvider.CHROMA: 'chroma'> collection_name='coll_emanual' persist_directory='./db/chroma_db'
Retrieval: type=<RetrievalType.SIMILARITY: 'similarity'> top_k=5
Generator: provider=<GeneratorProvider.LMSTUDIO: 'lmstudio'> model_name='google/gemma-4-e2b' temperature=0.0 base_url='${LMSTUDIO_BASE_URL}' api_key='${LMSTUDIO_API_KEY}'


In [4]:
loader = DataLoaderFactory.create(config.data_loader)
dataset = loader.load()
documents = dataset["documents"]

print(f"Loaded documents: {len(documents)}")
print(f"Sample document: {documents[0][:300] if documents else 'No documents found'}")


2026-07-03 23:33:14,243 | INFO	| real_world_rag | RAG System started
2026-07-03 23:33:14,245 | INFO	| real_world_rag | Loading local data...
2026-07-03 23:33:15,251 | DEBUG	| real_world_rag | Loader created: Dataset({     features: ['id', 'question', 'documents',
'response', 'generation_model_name', 'annotating_model_name', 'dataset_name',
'documents_sentences', 'response_sentences', 'sentence_support_information',
'unsupported_response_sentence_keys', 'adherence_score',
'overall_supported_explanation', 'relevance_explanation',
'all_relevant_sentence_keys', 'all_utilized_sentence_keys',
'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness',
'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance',
'gpt35_utilization', 'relevance_score', 'utilization_score',
'completeness_score'],     num_rows: 132 })
Loaded documents: 132
Sample document: ["Smart Hub. View descriptions of Smart Hub's basic functions. From Smart Hub, you can use the Internet search funct

In [5]:
chunker = ChunkingFactory.create(config.chunking)

all_chunks = []
all_chunk_metadata = []

for document_id, document in enumerate(documents, start=1):
    document_chunks = chunker.chunk(document)

    for chunk_id, chunk_group in enumerate(document_chunks, start=1):
        chunk_texts = chunk_group if isinstance(chunk_group, list) else [chunk_group]

        for chunk_text in chunk_texts:
            all_chunks.append(chunk_text)
            all_chunk_metadata.append({
                "document_id": document_id,
                "chunk_id": chunk_id,
                "chunk_index": len(all_chunks),
                "chunk_length": len(chunk_text),
            })

2026-07-03 23:33:15,275 | INFO	| real_world_rag | RAG System started
Initializing RecursiveChunker splitter_text=["Smart Hub. View descriptions of Smart Hub's basic functions. From Smart Hub, you can use the Internet search function, install and use various apps, view photos and videos, or listen to music stored on external storage devices, and perform more functions. Some Smart Hub services are paid services. To use Smart Hub , the TV must be connected to the Internet. Some Smart Hub features may not be supported depending on the service provider, language, or geographical area. Smart Hub service outages can be caused", 'on the service provider, language, or geographical area. Smart Hub service outages can be caused by disruptions in your Internet service. You can view the entire text of the Terms & Policy document by navigating to Settings Support Terms & Policy Try Now If you want to stop using Smart Hub , you can cancel the agreement. To cancel the Smart Hub service agreement, sele

In [6]:
embedder = EmbedderFactory.create(config.embedding)
embedding_sample = embedder.embed_query("What is retrieval augmented generation?")

print(f"Embedding dimension: {len(embedding_sample)}")
print(f"Embedding sample: {embedding_sample[:5]}")

Embedding dimension: 384
Embedding sample: [-0.038996633142232895, 0.01705986075103283, -0.03806664049625397, -0.023894181475043297, 0.0240192711353302]


In [7]:
vector_store = VectorStoreFactory.create(config.vector_store, embedder)

vector_store.add_documents(all_chunks, metadatas=all_chunk_metadata)

print("Vector store updated with chunk metadata")
print(f"Chunks stored: {len(all_chunks)}")


2026-07-03 23:33:20,020 | INFO	| real_world_rag | RAG System started
Initializing ChromaDBStore
db path: F:\Github\Personal\real-world-rag-system\db\chroma_db
Is db path exist: True


F:\Github\Personal\real-world-rag-system\src\rag_components\vector_stores\chroma_db_store.py:38: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_client = Chroma(


Vector store updated with chunk metadata
Chunks stored: 966


In [8]:
retriever = RetrieverFactory.create(
    config.retrieval,
    vector_store=vector_store,
    bm25_index=None,
)

query = "What is the use of search option in tne smart hub?"
retrieved = retriever.retrieve(query, k=config.retrieval.top_k)

print(f"Query: {query}")
print(f"Retrieved: {retrieved}")


Query: What is the use of search option in tne smart hub?
Retrieved: {'ids': [['doc_1071', 'doc_3263', 'doc_1140', 'doc_3332', 'doc_2203']], 'embeddings': None, 'documents': [['to between external devices connected to the Search You can search the apps or games in Smart Hub', 'to between external devices connected to the Search You can search the apps or games in Smart Hub', 'for channels, apps, titles of movies, or apps provided by the Smart Hub service. To use this', 'for channels, apps, titles of movies, or apps provided by the Smart Hub service. To use this', 'Smart Hub service agreement is required if you want to use Smart Hub and other features and']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'chunk_length': 97, 'document_id': 36, 'chunk_index': 1072, 'chunk_id': 1}, {'chunk_id': 1, 'chunk_index': 3264, 'chunk_length': 97, 'document_id': 102}, {'document_id': 36, 'chunk_index': 1141, 'chunk_id': 3, 'chunk_length': 92}, {'doc

In [9]:
generator_config = config.generator
generator_config.api_key = "lm-studio"
generator_config.base_url = "http://localhost:1234/v1"

generator = GeneratorFactory.create(generator_config)
print("✓ Generator created successfully")
print(f"Generator config: {generator_config}")

answer = generator.generate(    
    question=query,
    context=retrieved,
    system_prompt=ANOTHER_RAG_PROMPT,
)

print(f"Question: {query}")
print(f"Answer: {answer}")


✓ Generator created successfully
Generator config: provider=<GeneratorProvider.LMSTUDIO: 'lmstudio'> model_name='google/gemma-4-e2b' temperature=0.0 base_url='http://localhost:1234/v1' api_key='lm-studio'
Question: What is the use of search option in tne smart hub?
Answer: The search option in Smart Hub allows you to search for the following:

*   Apps or games in Smart Hub [Source 1]
*   Channels, apps, titles of movies, or apps provided by the Smart Hub service [Source 1]


## Evaluation

This section performs a lightweight notebook evaluation using the dataset reference answer and the retrieved context.

In [10]:
from pathlib import Path
import json
from datetime import datetime, timezone

from src.utils import helper


def normalize_text(value):
    return ' '.join(str(value).lower().split())

def token_set(value):
    text = normalize_text(value)
    return {token for token in text.split() if len(token) > 2}

def overlap_score(a, b):
    a_tokens = token_set(a)
    b_tokens = token_set(b)
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)

def context_hit_score(reference, retrieved_result):
    docs = retrieved_result.get('documents', [[]])[0] if isinstance(retrieved_result, dict) else []
    ref_text = normalize_text(reference)
    if not docs:
        return 0.0
    hits = sum(1 for doc in docs if ref_text[:80] in normalize_text(doc) or overlap_score(reference, doc) > 0.15)
    return hits / len(docs)

def save_results_to_json(unique_name, unique_name_config, evaluation_results):
    project_root = helper.get_project_root()

    report_record = {
        'unique_name': unique_name,
        'unique_name_config': unique_name_config,
        'datetime': datetime.now(timezone.utc).isoformat(),
        'scores': evaluation_results,
    }
    
    report_path = Path(project_root) / 'output' / 'report' / 'report_scores.json'
    report_path.parent.mkdir(parents=True, exist_ok=True)
    if report_path.exists():
        existing = json.loads(report_path.read_text(encoding='utf-8'))
        if not isinstance(existing, list):
            existing = [existing]
    else:
        existing = []

    existing.append(report_record)
    report_path.write_text(json.dumps(existing, indent=2), encoding='utf-8')

    
    print(report_record)
    print(f"Saved evaluation report to {report_path}")

def save_config_against_run(unique_name):
    project_root = helper.get_project_root()
    config_path = Path(project_root) / "output" / "report" / "config" / f"{unique_name}.json"
    config_path.parent.mkdir(parents=True, exist_ok=True)

    config_path.write_text(
        json.dumps(config.model_dump(mode="json"), indent=2),
        encoding="utf-8",
    )

    print(f"Saved config against run to {config_path}")


In [11]:
from datetime import datetime, timezone

sample = dataset[0]
evaluation_query = sample['question']
reference_answer = sample.get('response', '')
evaluation_retrieved = retriever.retrieve(evaluation_query, k=config.retrieval.top_k)
evaluation_answer = generator.generate(
    question=evaluation_query,
    context=evaluation_retrieved,
    system_prompt=BASIC_RAG_PROMPT,
)

retrieval_overlap = context_hit_score(reference_answer, evaluation_retrieved)
answer_overlap = overlap_score(reference_answer, evaluation_answer)
faithfulness_proxy = overlap_score(evaluation_answer, str(evaluation_retrieved))

evaluation_results = {
    'question': evaluation_query,
    'reference_answer': reference_answer,
    'generated_answer': evaluation_answer,
    'retrieval_overlap': retrieval_overlap,
    'answer_overlap': answer_overlap,
    'faithfulness_proxy': faithfulness_proxy,
}

unique_name_for_evaluation = f"evaluation_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}" 
unique_name_for_evaluation_config = f"config_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
save_results_to_json(unique_name_for_evaluation, unique_name_for_evaluation_config, evaluation_results)
save_config_against_run(unique_name_for_evaluation_config)


{'unique_name': 'evaluation_20260703_180447', 'unique_name_config': 'config_20260703_180447', 'datetime': '2026-07-03T18:04:47.627525+00:00', 'scores': {'question': 'What is the use of search option in tne smart hub?', 'reference_answer': 'The search option in Smart Hub allows users to search the Internet, install and use various apps, view photos and videos, listen to music stored on external storage devices, and perform more functions.', 'generated_answer': 'You can search the apps or games in Smart Hub between external devices connected to the Search [function] (document_1071, doc_3263, doc_1140, doc_3332, doc_2203).\n\nThe search can be used for:\n*   Channels\n*   Apps\n*   Titles of movies\n*   Apps provided by the Smart Hub service (document_1071, doc_3263, doc_1140, doc_3332, doc_2203).', 'retrieval_overlap': 0.8, 'answer_overlap': 0.1111111111111111, 'faithfulness_proxy': 0.1724137931034483}}
Saved evaluation report to F:\Github\Personal\real-world-rag-system\output\report\rep